# Comparare performanță pe testul din același set de date

**1. Descărcare dataset**

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive/test1/masks_test1

CNV  DRUSEN  NORMAL


**1. Încărcare model**

In [6]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_modelvar1_1_mobilnetv2.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/test1/masks_test1"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


**2. Testare**

**3. Afișare rezultate**

In [10]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 32 din 32
Corecte: 63 din 64
Corecte: 94 din 96
Corecte: 123 din 128
Corecte: 153 din 160
Corecte: 183 din 192
Corecte: 210 din 224
Corecte: 240 din 256
Corecte: 272 din 288
Corecte: 302 din 320
Corecte: 333 din 352
Corecte: 364 din 384
Corecte: 396 din 416
Corecte: 428 din 448
Corecte: 459 din 480
Corecte: 487 din 512
Corecte: 516 din 544
Corecte: 546 din 576
Corecte: 575 din 608
Corecte: 605 din 640
Corecte: 629 din 672
Corecte: 658 din 704
Corecte: 678 din 726
Acuratețe: 93.39%


In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.97      0.94      0.96       242
      DRUSEN       0.85      0.98      0.91       242
      NORMAL       1.00      0.89      0.94       242

    accuracy                           0.93       726
   macro avg       0.94      0.93      0.93       726
weighted avg       0.94      0.93      0.93       726


=== Acuratețe per clasă ===
CNV: 0.9380
DRUSEN: 0.9752
NORMAL: 0.8884


In [12]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_modelvar1_2_resnet18.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [13]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/test1/masks_test1"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [14]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 32 din 32
Corecte: 63 din 64
Corecte: 93 din 96
Corecte: 123 din 128
Corecte: 153 din 160
Corecte: 185 din 192
Corecte: 213 din 224
Corecte: 243 din 256
Corecte: 275 din 288
Corecte: 307 din 320
Corecte: 338 din 352
Corecte: 368 din 384
Corecte: 399 din 416
Corecte: 431 din 448
Corecte: 461 din 480
Corecte: 491 din 512
Corecte: 519 din 544
Corecte: 550 din 576
Corecte: 580 din 608
Corecte: 609 din 640
Corecte: 635 din 672
Corecte: 667 din 704
Corecte: 688 din 726
Acuratețe: 94.77%


In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.97      0.95      0.96       242
      DRUSEN       0.88      0.97      0.93       242
      NORMAL       1.00      0.92      0.96       242

    accuracy                           0.95       726
   macro avg       0.95      0.95      0.95       726
weighted avg       0.95      0.95      0.95       726


=== Acuratețe per clasă ===
CNV: 0.9504
DRUSEN: 0.9711
NORMAL: 0.9215
